# 01 · Exploratory Data Analysis

**Goal:** Understand the raw dataset before any modelling — revenue trends,
top products, geographic spread, and customer spend distribution.

| Step | Description |
|------|-------------|
| 1 | Load & inspect raw data |
| 2 | Missing value audit |
| 3 | Revenue over time |
| 4 | Top products & countries |
| 5 | Customer-level spend distribution |
| 6 | Correlation heat-map |

## 0 · Imports & config

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))   # project root

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_and_clean, get_summary

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
PALETTE = ["#4361EE", "#3A0CA3", "#7209B7", "#F72585", "#4CC9F0"]
sns.set_theme(style="whitegrid", palette=PALETTE)


## 1 · Load raw data

In [ ]:
DATA_PATH = "../data/Online_Retail.xlsx"
df_raw = pd.read_excel(DATA_PATH)
print(f"Shape  : {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


## 2 · Missing value audit

In [ ]:
missing = df_raw.isnull().sum().rename("missing")
pct     = (missing / len(df_raw) * 100).rename("pct_%").round(2)
pd.concat([missing, pct], axis=1).query("missing > 0")


In [ ]:
# CustomerID is the most important — visualise how much we'd lose
fig, ax = plt.subplots(figsize=(6, 3))
labels  = ["Has CustomerID", "Missing CustomerID"]
sizes   = [df_raw["CustomerID"].notna().sum(), df_raw["CustomerID"].isna().sum()]
ax.barh(labels, sizes, color=[PALETTE[0], "#e63946"])
ax.set_xlabel("Row count")
ax.set_title("CustomerID completeness")
for i, v in enumerate(sizes):
    ax.text(v + 500, i, f"{v:,}  ({v/len(df_raw)*100:.1f}%)", va="center")
plt.tight_layout()
plt.show()


## 3 · Clean data

In [ ]:
df = load_and_clean(DATA_PATH)
summary = get_summary(df)
pd.Series(summary)


## 4 · Revenue trend over time

In [ ]:
monthly = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))["TotalPrice"]
      .sum()
      .reset_index()
)
monthly["InvoiceDate"] = monthly["InvoiceDate"].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly["InvoiceDate"], monthly["TotalPrice"] / 1e3,
        marker="o", color=PALETTE[0], linewidth=2.5, markersize=6)
ax.fill_between(monthly["InvoiceDate"], monthly["TotalPrice"] / 1e3,
                alpha=0.08, color=PALETTE[0])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:.0f}k"))
ax.tick_params(axis="x", rotation=45)
ax.set_title("Monthly Revenue")
ax.set_ylabel("Revenue (£k)")
plt.tight_layout()
plt.savefig("../reports/figures/eda_monthly_revenue.png", bbox_inches="tight")
plt.show()
print("Peak month:", monthly.loc[monthly['TotalPrice'].idxmax(), 'InvoiceDate'])


## 5 · Day-of-week & hour patterns

In [ ]:
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"]      = df["InvoiceDate"].dt.hour

day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
day_rev = df.groupby("DayOfWeek")["TotalPrice"].sum().reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(day_rev.index, day_rev.values / 1e3, color=PALETTE[1])
axes[0].set_title("Revenue by Day of Week")
axes[0].set_ylabel("Revenue (£k)")
axes[0].tick_params(axis="x", rotation=30)

hour_rev = df.groupby("Hour")["TotalPrice"].sum()
axes[1].bar(hour_rev.index, hour_rev.values / 1e3, color=PALETTE[2])
axes[1].set_title("Revenue by Hour of Day")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Revenue (£k)")

plt.tight_layout()
plt.savefig("../reports/figures/eda_time_patterns.png", bbox_inches="tight")
plt.show()


## 6 · Top products

In [ ]:
top_products = (
    df.groupby("Description")["TotalPrice"]
      .sum()
      .sort_values(ascending=False)
      .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
top_products[::-1].plot(kind="barh", ax=ax, color=PALETTE[1])
ax.set_title("Top 15 Products by Revenue")
ax.set_xlabel("Total Revenue (£)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e3:.0f}k"))
plt.tight_layout()
plt.savefig("../reports/figures/eda_top_products.png", bbox_inches="tight")
plt.show()


## 7 · Country-wise revenue (ex-UK)

In [ ]:
country_rev = (
    df[df["Country"] != "United Kingdom"]
      .groupby("Country")["TotalPrice"]
      .sum()
      .sort_values(ascending=False)
      .head(12)
)

fig, ax = plt.subplots(figsize=(10, 5))
country_rev.plot(kind="bar", ax=ax, color=PALETTE[3])
ax.set_title("Revenue by Country (excl. UK) — Top 12")
ax.set_ylabel("Revenue (£)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig("../reports/figures/eda_country_revenue.png", bbox_inches="tight")
plt.show()


## 8 · Customer spend distribution

In [ ]:
cust_spend = df.groupby("CustomerID")["TotalPrice"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution clipped at 95th percentile
p95 = cust_spend.quantile(0.95)
axes[0].hist(cust_spend[cust_spend <= p95], bins=60,
             color=PALETTE[0], edgecolor="white", linewidth=0.3)
axes[0].set_title(f"Spend Distribution (≤ £{p95:,.0f} — 95th pct)")
axes[0].set_xlabel("Total Spend (£)")
axes[0].set_ylabel("# Customers")

# Log scale version (shows full range)
axes[1].hist(np.log1p(cust_spend), bins=60,
             color=PALETTE[4], edgecolor="white", linewidth=0.3)
axes[1].set_title("Log(1 + Spend) Distribution")
axes[1].set_xlabel("Log(1 + Total Spend)")

plt.tight_layout()
plt.savefig("../reports/figures/eda_spend_distribution.png", bbox_inches="tight")
plt.show()

print(cust_spend.describe().apply(lambda x: f"£{x:,.2f}"))


## 9 · Correlation heat-map

In [ ]:
num_cols = ["Quantity", "UnitPrice", "TotalPrice"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Heat-map")
plt.tight_layout()
plt.savefig("../reports/figures/eda_correlation.png", bbox_inches="tight")
plt.show()


## 10 · Summary takeaways

- **Revenue spikes in Nov–Dec** — seasonality is strong; models trained without
  time-based splits may overfit to this period.
- **Thursdays and midday (12–14h)** are peak trading windows.
- **Customer spend is highly right-skewed** — a small number of customers
  dominate total revenue. Log-transforming monetary features will be essential.
- **Netherlands, Germany, France** are the top international markets after UK.
- **UK dominates** (>85% of revenue) — country is a weak feature for customer ML.